# Oil-Price Transmission Analysis – ARDL & NARDL

**Mohammed Tukur Saidu, PhD**  
Applied Economist | Energy & Macro-Finance  

This notebook demonstrates the core modelling pipeline used in my research on oil-price transmission in South Africa, Morocco and Côte d’Ivoire.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_prep import load_and_prepare
from ardl_models import estimate_ardl, estimate_nardl, residual_diagnostics
from diagnostics import plot_series, plot_partial_sums, cusum_plot

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

## 1. Load & prepare data

In [ ]:
df = load_and_prepare("../data/sample_macro_africa.csv")
print(df.shape)
df.head()

## 2. Visual inspection

In [ ]:
plot_series(df, ["lbrent", "lcpi_sa", "lreer_sa"],
            title="Log Brent, Log CPI (SA) & Log REER (SA)")

In [ ]:
plot_partial_sums(df)

## 3. Symmetric ARDL – Inflation equation (South Africa example)

In [ ]:
endog = df["lcpi_sa"]
exog  = df[["lbrent", "lreer_sa"]]

ardl_res = estimate_ardl(endog, exog, maxlag=4)
print(ardl_res.summary())

In [ ]:
print("Residual diagnostics:")
print(residual_diagnostics(ardl_res))

## 4. Asymmetric NARDL

In [ ]:
nardl_res = estimate_nardl(
    endog=df["lcpi_sa"],
    brent_pos=df["brent_pos"],
    brent_neg=df["brent_neg"],
    controls=df[["lreer_sa"]],
    maxlag=4
)
print(nardl_res.summary())

## 5. Stability check

In [ ]:
cusum_plot(nardl_res)

---
**Notes for users**
- Replace the sample data with World Bank, IMF IFS or national statistical office series for real research.
- Full bounds-test critical values and long-run coefficient standard errors can be obtained via the `bounds` package or by implementing Pesaran et al. (2001) tables.
- This pipeline was used (with real data) in the PhD dissertation and subsequent journal articles.